# Klasifikasi Kematangan Tapai - Recurrent Neural Network (RNN/LSTM)

Notebook ini menggunakan **LSTM (Long Short-Term Memory)** — sebuah arsitektur RNN — dari TensorFlow/Keras untuk memprediksi status kematangan tapai.

## Mengapa RNN/LSTM?

Dataset kematangan tapai adalah data **deret waktu (time series)**:
- Setiap percobaan memiliki 60 titik waktu berurutan (jam ke-1 s/d jam ke-60)
- Status kematangan di jam ke-*t* **sangat dipengaruhi** oleh kondisi jam-jam sebelumnya
- Model klasik (Logistic Regression, SVM, RF) **mengabaikan urutan** ini
- **LSTM** secara eksplisit memodelkan dependensi temporal jangka panjang melalui mekanisme *cell state* dan *gates*

## Dataset
- **Input sequence:** urutan `[suhu, kelembaban, kadar_gas]` per jam dalam satu percobaan
- **Target:** `status_kematangan` untuk setiap timestep
- **10 percobaan** = 10 sequence, masing-masing 60 timestep

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

plt.style.use('seaborn-v0_8')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

ModuleNotFoundError: No module named 'tensorflow'

## 1. Load Data

In [ ]:
df = pd.read_csv('../dataset/dataset_kematangan_tapai_v2.csv')
print('Shape:', df.shape)
print('Percobaan unik:', sorted(df['percobaan'].unique()))
df.head(10)

## 2. Preprocessing - Membentuk Sequence untuk RNN

Berbeda dengan model klasik, untuk RNN kita perlu membentuk data dalam format **[samples, timesteps, features]**.

Kita gunakan pendekatan **sliding window** agar model bisa belajar dari konteks beberapa jam sebelumnya.

In [ ]:
label_map = {'belum matang': 0, 'matang': 1, 'terlalu matang': 2}
df['label'] = df['status_kematangan'].map(label_map)

feature_cols = ['suhu', 'kelembaban', 'kadar_gas']
WINDOW_SIZE = 10  # menggunakan 10 jam sebelumnya untuk prediksi
N_CLASSES = 3

# Normalisasi
scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

def create_sequences(group_df, window_size=10):
    """Buat sliding window sequences dari satu percobaan."""
    X_seq, y_seq = [], []
    values = group_df[feature_cols].values
    labels = group_df['label'].values
    for i in range(window_size, len(values)):
        X_seq.append(values[i - window_size:i])
        y_seq.append(labels[i])
    return np.array(X_seq), np.array(y_seq)

# Bagi percobaan: 1-8 untuk train, 9-10 untuk test
train_percobaan = [1, 2, 3, 4, 5, 6, 7, 8]
test_percobaan  = [9, 10]

X_train_list, y_train_list = [], []
for p in train_percobaan:
    grp = df[df['percobaan'] == p].sort_values('jam')
    Xs, ys = create_sequences(grp, WINDOW_SIZE)
    X_train_list.append(Xs)
    y_train_list.append(ys)

X_test_list, y_test_list = [], []
for p in test_percobaan:
    grp = df[df['percobaan'] == p].sort_values('jam')
    Xs, ys = create_sequences(grp, WINDOW_SIZE)
    X_test_list.append(Xs)
    y_test_list.append(ys)

X_train = np.concatenate(X_train_list)
y_train = np.concatenate(y_train_list)
X_test  = np.concatenate(X_test_list)
y_test  = np.concatenate(y_test_list)

y_train_cat = to_categorical(y_train, N_CLASSES)
y_test_cat  = to_categorical(y_test, N_CLASSES)

print(f'X_train: {X_train.shape}  →  [samples, timesteps, features]')
print(f'X_test : {X_test.shape}')
print(f'y_train: {y_train_cat.shape}')

## 3. Arsitektur Model LSTM

In [ ]:
def build_lstm_model(input_shape, n_classes):
    model = Sequential([
        # Layer LSTM pertama - menangkap pola temporal jangka pendek
        Bidirectional(
            LSTM(64, return_sequences=True),
            input_shape=input_shape
        ),
        BatchNormalization(),
        Dropout(0.3),

        # Layer LSTM kedua - menangkap pola temporal jangka panjang
        LSTM(32, return_sequences=False),
        BatchNormalization(),
        Dropout(0.2),

        # Fully connected layers
        Dense(32, activation='relu'),
        Dropout(0.1),

        # Output layer
        Dense(n_classes, activation='softmax')
    ])
    return model

model = build_lstm_model(
    input_shape=(WINDOW_SIZE, len(feature_cols)),
    n_classes=N_CLASSES
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train_cat,
    epochs=150,
    batch_size=16,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

## 5. Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', color='#3498db')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='#e74c3c', linestyle='--')
axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Acc', color='#3498db')
axes[1].plot(history.history['val_accuracy'], label='Val Acc', color='#e74c3c', linestyle='--')
axes[1].set_title('Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('LSTM Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_rnn_history.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Evaluasi pada Data Test

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

target_names = ['belum matang', 'matang', 'terlalu matang']
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - LSTM (RNN)\nAccuracy: {accuracy*100:.2f}%')
plt.tight_layout()
plt.savefig('plot_rnn_cm.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Prediksi Urutan Kematangan (Per Percobaan)

In [ ]:
label_inv = {0: 'belum matang', 1: 'matang', 2: 'terlalu matang'}
colors_map = {0: '#3498db', 1: '#2ecc71', 2: '#e74c3c'}

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

for idx, (p, ax) in enumerate(zip(test_percobaan, axes)):
    X_p, y_p = X_test_list[idx], y_test_list[idx]
    y_p_pred = np.argmax(model.predict(X_p, verbose=0), axis=1)
    
    jam_range = range(WINDOW_SIZE + 1, WINDOW_SIZE + 1 + len(y_p))
    
    ax.scatter(jam_range, y_p, c=[colors_map[v] for v in y_p],
               label='Actual', marker='o', s=60, zorder=3)
    ax.scatter(jam_range, y_p_pred, c=[colors_map[v] for v in y_p_pred],
               label='Predicted', marker='x', s=80, linewidths=2, zorder=4)
    
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Belum Matang', 'Matang', 'Terlalu Matang'])
    ax.set_xlabel('Jam ke-')
    ax.set_title(f'Percobaan {p} - Prediksi LSTM')
    
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker='o', color='gray', label='Actual', markersize=8),
        Line2D([0], [0], marker='x', color='gray', label='Predicted', markersize=8, linewidth=2)
    ]
    ax.legend(handles=handles)

plt.suptitle('Prediksi LSTM per Percobaan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_rnn_prediction.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Mengapa LSTM Lebih Unggul untuk Dataset Ini?

| Aspek | Model Klasik (LR, SVM, RF) | LSTM (RNN) |
|-------|---------------------------|------------|
| Memori temporal | ❌ Tidak ada | ✅ Melalui cell state & gates |
| Dependensi antar jam | ❌ Diabaikan | ✅ Dipelajari secara eksplisit |
| Input format | Flat vector per jam | Sequence [timesteps × features] |
| Cocok untuk time series | ❌ Tidak dirancang | ✅ Dirancang khusus |
| Interpretabilitas | ✅ Lebih mudah | ⚠️ Black box |

**Kesimpulan:** Proses fermentasi tapai adalah proses kimia-biologis yang bersifat **kumulatif**. Kondisi suhu, kelembaban, dan kadar gas pada jam-jam sebelumnya mempengaruhi kondisi saat ini. LSTM mampu menangkap dinamika temporal ini sehingga memberikan prediksi yang lebih akurat dan lebih realistis secara ilmiah.